# 家計調査データ取得(Google Colab版)

**事前準備(このノートブックを開く前に1回だけ):**

1. 左サイドバーの🔑(Secrets)アイコンをクリック
2. 「新しいシークレットを追加」→ 名前 `ESTAT_APP_ID` / 値に自分のappIdを貼り付け
3. 「ノートブックからのアクセス」トグルをON

これでappIdをコードに直書きせずに使えます。

In [ ]:
import requests
import pandas as pd

try:
    from google.colab import userdata
    APP_ID = userdata.get("ESTAT_APP_ID")
except ImportError:
    # Colab以外(ローカル)で開いた場合は環境変数から読む
    import os
    APP_ID = os.environ.get("ESTAT_APP_ID", "")

if not APP_ID:
    raise RuntimeError("ESTAT_APP_ID が取得できません。Secretsの設定を確認してください。")

BASE_URL = "https://api.e-stat.go.jp/rest/3.0/app/json"
KAKEI_GOV_STATS_CODE = "00200561"  # 家計調査の政府統計コード
print("appId 読み込みOK")

## 1. 統計表を検索する

In [ ]:
def search_stats_list(search_word: str, limit: int = 30) -> pd.DataFrame:
    params = {
        "appId": APP_ID,
        "statsCode": KAKEI_GOV_STATS_CODE,
        "searchWord": search_word,
        "limit": limit,
    }
    res = requests.get(f"{BASE_URL}/getStatsList", params=params, timeout=30)
    res.raise_for_status()
    body = res.json()["GET_STATS_LIST"]
    result = body["RESULT"]
    if result["STATUS"] != 0:
        raise RuntimeError(f"e-Stat APIエラー: {result.get('ERROR_MSG')}")
    table_inf = body["DATALIST_INF"].get("TABLE_INF", [])
    if isinstance(table_inf, dict):
        table_inf = [table_inf]
    rows = [
        {
            "statsDataId": t["@id"],
            "title": t["TITLE"].get("$", t["TITLE"]) if isinstance(t["TITLE"], dict) else t["TITLE"],
            "survey_date": t.get("SURVEY_DATE"),
        }
        for t in table_inf
    ]
    return pd.DataFrame(rows)


search_stats_list("貯蓄")

## 2. メタ情報(分類軸)を確認する

In [ ]:
def get_class_map(stats_data_id: str, axis_id: str) -> dict:
    params = {"appId": APP_ID, "statsDataId": stats_data_id}
    res = requests.get(f"{BASE_URL}/getMetaInfo", params=params, timeout=30)
    body = res.json()["GET_META_INFO"]
    if body["RESULT"]["STATUS"] != 0:
        raise RuntimeError(f"e-Stat APIエラー: {body['RESULT'].get('ERROR_MSG')}")
    class_obj = body["METADATA_INF"]["CLASS_INF"]["CLASS_OBJ"]
    for c in class_obj:
        if c["@id"] == axis_id:
            classes = c["CLASS"]
            if isinstance(classes, dict):
                classes = [classes]
            return {cl["@code"]: cl["@name"] for cl in classes}
    return {}


# 例: STATS_DATA_ID = "0002210009"
# get_class_map(STATS_DATA_ID, "cat01")

## 3. 実データを取得してDataFrame化する

In [ ]:
def get_stats_data(stats_data_id: str, **extra_params) -> pd.DataFrame:
    params = {"appId": APP_ID, "statsDataId": stats_data_id, **extra_params}
    res = requests.get(f"{BASE_URL}/getStatsData", params=params, timeout=60)
    res.raise_for_status()
    body = res.json()["GET_STATS_DATA"]
    if body["RESULT"]["STATUS"] != 0:
        raise RuntimeError(f"e-Stat APIエラー: {body['RESULT'].get('ERROR_MSG')}")
    values = body["STATISTICAL_DATA"]["DATA_INF"]["VALUE"]
    return pd.DataFrame(values)


# 例: 貯蓄・負債（年間収入階級別、二人以上の世帯）
df = get_stats_data("0002210009", cdCat01="011,012,028", cdCat02="03")

# 分類コードを名前に変換
cat01_map = get_class_map("0002210009", "cat01")
cat03_map = get_class_map("0002210009", "cat03")
df["項目"] = df["@cat01"].astype(str).str.zfill(3).map(cat01_map)
df["年間収入階級"] = df["@cat03"].astype(str).str.zfill(3).map(cat03_map)
df_out = df[["項目", "年間収入階級", "@time", "$", "@unit"]].rename(
    columns={"@time": "時期", "$": "値", "@unit": "単位"}
)
df_out.head(10)

## 4. CSVとして保存(Googleドライブに永続化)

Colabのランタイムはセッション終了時に消えるので、残したいファイルはGoogleドライブに保存する。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
SAVE_DIR = "/content/drive/MyDrive/kakei_data"
os.makedirs(SAVE_DIR, exist_ok=True)

df_out.to_csv(f"{SAVE_DIR}/kakei_savings_debt_by_income.csv", index=False, encoding="utf-8-sig")
print(f"保存先: {SAVE_DIR}/kakei_savings_debt_by_income.csv")

## 5. 可視化(例)

In [ ]:
import matplotlib.pyplot as plt

pivot = df_out[df_out["項目"] == "貯蓄"].pivot_table(
    index="時期", columns="年間収入階級", values="値"
)
pivot.plot(figsize=(10, 5), title="年間収入階級別 貯蓄現在高の推移")
plt.ylabel("万円")
plt.show()